# Rubrik adapter'ı — adım maliyeti ölçümü

Bu notebook eğitmiyor. **Tek bir sayıyı** ölçüyor: bir satırın ileri+geri
geçişi kaç saniye sürüyor, ve bu sayı Kaggle'ın verdiği makinede kaç GPU
göründüğüne bağlı mı.

Sebep: `rubric-qlora`'nın iptal edilen koşusu OOM değildi. Temiz eğitiyordu,
**adım başına ~910 saniye** — 11 saat 24 dakikada 150 adımın 45'i, sonra 12
saatlik oturum duvarı. Runbook o notebook için "~5,5 saat" diyor; o tahmin
Gemma-2-2B hattından geldi ve Qwen3-4B'ye taşınmadı.

Şüphe, logdaki bir tutarsızlıktan: `train_qlora_qwen.py` kendi aritmetiğiyle
`~300 optimizer adımı` bastı, Trainer'ın progress bar'ı ise `150`'ye koştu.
Trainer'ın adım sayısı tam olarak iki cihaz gördüğünde yarıya iner
(`per_device × accum × world_size` = 1×16×2 = 32). Yani `machine_shape:
NvidiaTeslaT4` iki kart veriyor olabilir, ve Trainer `device_map={"": 0}` ile
tek karta sabitlenmiş 4-bit modeli `DataParallel`'e sarıyor: kuantize
ağırlıklar her microbatch'te replike ediliyor.

Bu bir hipotez. 12 saat daha yakıp doğrulamak yerine burada 25 dakikada
ölçülüyor, ve ölçüm **`train_qlora_qwen.py`'nin kendisini** koşuyor — elle
yazılmış bir eğitim döngüsü `DataParallel` yolunu hiç kurmaz, yani ölçmek
istediğimiz şeyi ölçmezdi.

`machine_shape` bilerek train kernel'iyle **aynı** bırakıldı. Sorulan soru o
shape'in ne verdiği.

In [ ]:
# The datum this notebook exists for. Everything below is here to price it.
import os, torch

print("device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cap = torch.cuda.get_device_capability(i)
    print(f"  [{i}] {p.name}  sm_{cap[0]}{cap[1]}  {p.total_memory / 1024**3:.1f} GB")

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))

# If this prints 2, the hypothesis has its mechanism and regime B below is the
# fix. If it prints 1, the 910 s/step is something else and the two regimes
# will come out the same — which is also an answer, just a more expensive one.

In [ ]:
# Same pins as the training notebook. A probe on different library versions
# prices a different run.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)

In [ ]:
import glob, json, shutil, sys


def find_mount(slug, marker):
    """Locate one input mount by slug, not by filename. Copied deliberately
    from the training notebook: a probe that resolves its inputs differently
    from the run it is pricing can be measuring a different file."""
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, f"'{slug}' bagli degil (aranan: {marker})"
    return os.path.dirname(sorted(hits, key=len)[0])


WORK = "/kaggle/working"
DATA = find_mount("emrahik/rubric-dataset", "rubric_train.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# 8 rows, sampled at a fixed stride rather than head -8: sequence length drives
# the cost being measured, and the front of the file is not a random sample of
# it. Stride sampling keeps the length distribution close to the full set's.
PROBE_ROWS = 8
EVAL_ROWS = 4

full = [l for l in open("data/rubric_train.jsonl", encoding="utf-8") if l.strip()]
stride = max(1, len(full) // PROBE_ROWS)
probe = [full[i * stride] for i in range(PROBE_ROWS)]
with open("data/probe_train.jsonl", "w", encoding="utf-8") as fh:
    fh.writelines(probe)

full_eval = [l for l in open("data/rubric_eval.jsonl", encoding="utf-8") if l.strip()]
stride_e = max(1, len(full_eval) // EVAL_ROWS)
with open("data/probe_eval.jsonl", "w", encoding="utf-8") as fh:
    fh.writelines([full_eval[i * stride_e] for i in range(EVAL_ROWS)])

print(f"probe: {PROBE_ROWS} train / {EVAL_ROWS} eval rows"
      f"  (full set: {len(full)} / {len(full_eval)})")

In [ ]:
import subprocess, time

# grad-accum 1 so one optimizer step is one row: the atomic cost, without the
# 16x amplification that made the real run's step read 910 s.
BASE_ARGS = [sys.executable, "train_qlora_qwen.py",
             "--train", "data/probe_train.jsonl",
             "--eval", "data/probe_eval.jsonl",
             "--max-seq-len", "2560",
             "--epochs", "1", "--grad-accum", "1"]

TRAIN_ENV = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True",
                 PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")


def run(label, extra_env, out_dir):
    """Run the real training script once and return wall seconds.

    subprocess, not an import: CUDA_VISIBLE_DEVICES is read when CUDA
    initialises, so the two regimes cannot exist in one process.
    """
    env = dict(TRAIN_ENV, **extra_env)
    print(f"\n===== {label} =====", flush=True)
    t0 = time.time()
    r = subprocess.run(BASE_ARGS + ["--out-dir", out_dir], env=env)
    dt = time.time() - t0
    print(f"\n{label}: exit {r.returncode}, wall {dt / 60:.1f} min", flush=True)
    return dt, r.returncode

In [ ]:
# Regime A — exactly what the cancelled run did. No device pinning.
wall_a, rc_a = run("A: as the real run (no pinning)", {}, "out/probe_a")

In [ ]:
# Regime B — one GPU visible. If Trainer was wrapping the pinned 4-bit model in
# DataParallel, this is where it stops.
wall_b, rc_b = run("B: CUDA_VISIBLE_DEVICES=0", {"CUDA_VISIBLE_DEVICES": "0"}, "out/probe_b")

## Sonuç

İki koşu aynı 8 satırı, aynı kütüphane sürümleriyle, aynı oturumda geçti; tek
fark görünen GPU sayısı. Model yükleme maliyeti ikisinde de aynı olduğu için
**aradaki fark** adım maliyetine ait.

Aşağıdaki projeksiyon tam koşuyu fiyatlıyor: 1600 satır × 3 epoch = 4800 satır
geçişi. 12 saatin altına inen bir konfigürasyon varsa, train kernel'i o
konfigürasyonla push'lanır.

In [ ]:
FULL_ROW_PASSES = 1600 * 3
LOAD_GUESS_S = 180  # model download + 4-bit load, identical in both regimes


def report(label, wall, rc):
    if rc != 0:
        print(f"{label}: FAILED (exit {rc}) — projeksiyon yok")
        return None
    per_row = max(0.0, wall - LOAD_GUESS_S) / PROBE_ROWS
    hours = per_row * FULL_ROW_PASSES / 3600
    print(f"{label}")
    print(f"  wall              {wall / 60:6.1f} min ({PROBE_ROWS} satir)")
    print(f"  ~s/satir          {per_row:6.1f}  (yaklasik {LOAD_GUESS_S}s yukleme dusuldu)")
    print(f"  tam kosu tahmini  {hours:6.1f} saat   -> 12 saatlik oturuma "
          f"{'SIGAR' if hours < 11 else 'SIGMAZ'}")
    return per_row


print(f"device_count: {torch.cuda.device_count()}\n")
a = report("A: pinlemesiz", wall_a, rc_a)
b = report("B: CUDA_VISIBLE_DEVICES=0", wall_b, rc_b)

if a and b:
    print(f"\nB, A'nin {a / b:.1f}x hizinda.")
    if a / b > 1.5:
        print("Hipotez dogrulandi: cihaz sayisi adim maliyetini belirliyor. "
              "Train notebook'u CUDA_VISIBLE_DEVICES=0 ile push'lanmali.")
    else:
        print("Hipotez yanlis: iki rejim ayni. 910 s/adim baska bir seyden "
              "geliyor ve siradaki sucluler NF4 dequant yolu ile "
              "gradient checkpointing. Kapsami kirpmadan once onlar olculmeli.")